In [1]:
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
from langchain.agents import AgentState

class CustomState(AgentState):
    favourite_colour: str

## Write to state

In [3]:
from langchain.tools import tool, ToolRuntime
from langgraph.types import Command
from langchain.messages import ToolMessage

@tool
def update_favourite_colour(favourite_colour: str, runtime: ToolRuntime) -> Command:
    """Update the favourite colour of the user in the state once they've revealed it."""
    return Command(update={
        "favourite_colour": favourite_colour, 
        "messages": [ToolMessage("Successfully updated favourite colour", tool_call_id=runtime.tool_call_id)]}
        )

In [4]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver
from langchain_openai import ChatOpenAI


model = ChatOpenAI(
    model="qwen3:4b",
    api_key="ollama",
    base_url="http://localhost:11434/v1"
)

agent = create_agent(
    model,
    tools=[update_favourite_colour],
    checkpointer=InMemorySaver(),
    state_schema=CustomState
)

In [5]:
from langchain.messages import HumanMessage

response = agent.invoke(
    { "messages": [HumanMessage(content="My favourite colour is green")]},
    {"configurable": {"thread_id": "1"}}
)

In [6]:
from pprint import pprint

pprint(response)

{'favourite_colour': 'green',
 'messages': [HumanMessage(content='My favourite colour is green', additional_kwargs={}, response_metadata={}, id='9f0d669f-32b1-4bbf-ba86-70598b307195'),
              AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 236, 'prompt_tokens': 151, 'total_tokens': 387, 'completion_tokens_details': None, 'prompt_tokens_details': None}, 'model_provider': 'openai', 'model_name': 'qwen3:4b', 'system_fingerprint': 'fp_ollama', 'id': 'chatcmpl-444', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a0389b-0c35-7391-a5d7-520d0bc25df0-0', tool_calls=[{'name': 'update_favourite_colour', 'args': {'favourite_colour': 'green'}, 'id': 'call_x6c6umc0', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 151, 'output_tokens': 236, 'total_tokens': 387, 'input_token_details': {}, 'output_token_details': {}}),
              ToolMessage(content='Successfully updated favourit

In [7]:
response = agent.invoke(
    { 
        "messages": [HumanMessage(content="Hello, how are you?")],
        "favourite_colour": "green"
    },
    {"configurable": {"thread_id": "10"}}
)

pprint(response)

{'favourite_colour': 'green',
 'messages': [HumanMessage(content='Hello, how are you?', additional_kwargs={}, response_metadata={}, id='0fda1bb3-c35a-4fa0-af1b-02c4e92cc03a'),
              AIMessage(content="Hello! I'm doing great. How can I assist you today? 😊", additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 245, 'prompt_tokens': 152, 'total_tokens': 397, 'completion_tokens_details': None, 'prompt_tokens_details': None}, 'model_provider': 'openai', 'model_name': 'qwen3:4b', 'system_fingerprint': 'fp_ollama', 'id': 'chatcmpl-373', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a0389b-1f99-7ef0-a815-98965065c395-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 152, 'output_tokens': 245, 'total_tokens': 397, 'input_token_details': {}, 'output_token_details': {}})]}


## Read state

In [8]:
@tool
def read_favourite_colour(runtime: ToolRuntime) -> str:
    """Read the favourite colour of the user from the state."""
    try:
        return runtime.state["favourite_colour"]
    except KeyError:
        return "No favourite colour found in state"

agent = create_agent(
    model,
    tools=[update_favourite_colour, read_favourite_colour],
    checkpointer=InMemorySaver(),
    state_schema=CustomState
)

In [9]:
response = agent.invoke(
    { "messages": [HumanMessage(content="My favourite colour is green")]},
    {"configurable": {"thread_id": "1"}}
)

pprint(response)

{'favourite_colour': 'green',
 'messages': [HumanMessage(content='My favourite colour is green', additional_kwargs={}, response_metadata={}, id='a1b1cab5-5f43-448f-8b3b-3203b59d0366'),
              AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 234, 'prompt_tokens': 188, 'total_tokens': 422, 'completion_tokens_details': None, 'prompt_tokens_details': None}, 'model_provider': 'openai', 'model_name': 'qwen3:4b', 'system_fingerprint': 'fp_ollama', 'id': 'chatcmpl-861', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a0389b-4bc2-75e1-9c0d-087d032d4148-0', tool_calls=[{'name': 'update_favourite_colour', 'args': {'favourite_colour': 'green'}, 'id': 'call_9u9ei8yi', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 188, 'output_tokens': 234, 'total_tokens': 422, 'input_token_details': {}, 'output_token_details': {}}),
              ToolMessage(content='Successfully updated favourit

In [10]:
response = agent.invoke(
    { "messages": [HumanMessage(content="What's my favourite colour?")]},
    {"configurable": {"thread_id": "1"}}
)

pprint(response)

{'favourite_colour': 'green',
 'messages': [HumanMessage(content='My favourite colour is green', additional_kwargs={}, response_metadata={}, id='a1b1cab5-5f43-448f-8b3b-3203b59d0366'),
              AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 234, 'prompt_tokens': 188, 'total_tokens': 422, 'completion_tokens_details': None, 'prompt_tokens_details': None}, 'model_provider': 'openai', 'model_name': 'qwen3:4b', 'system_fingerprint': 'fp_ollama', 'id': 'chatcmpl-861', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a0389b-4bc2-75e1-9c0d-087d032d4148-0', tool_calls=[{'name': 'update_favourite_colour', 'args': {'favourite_colour': 'green'}, 'id': 'call_9u9ei8yi', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 188, 'output_tokens': 234, 'total_tokens': 422, 'input_token_details': {}, 'output_token_details': {}}),
              ToolMessage(content='Successfully updated favourit